In [6]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier

This document implements a centralized logistic regression model for a single complication.

The implementation is very similar to `logistic_regression_FL_one_complication`, but instead of distributing the data across clients and aggregating local models, all training data are pooled together and used to train a single global model.

## Data

In [7]:
path_train = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\raw\synthetic_dataset_A_non-iid.csv"
path_test = r"C:\Users\oskar\OneDrive\Desktop\4 Semester\Dataproject\Federated-dental-risk-vol2\federated-dental-risk-prediction\data\processed\A\global_test_set_non-iid.csv"

df = pd.read_csv(path_train)
test_data = pd.read_csv(path_test)

"""
df = (
    df.groupby("Client", group_keys=False)
      .apply(lambda x: x.sample(frac=20000 / len(df), random_state=42))
      .reset_index(drop=True)
)
"""


variables = df.columns[2:27].tolist()

# Ændre Target_col for øvrige complicationer.

target_col = "Risk_NerveDysesthesia"
category_col = "Risk_Category_NerveDysesthesia"
client_col = "Client"


scaler = StandardScaler()

train_design = pd.DataFrame(
    scaler.fit_transform(df[variables]),
    columns=variables,
    index=df.index
)

test_design = pd.DataFrame(
    scaler.transform(test_data[variables]),
    columns=variables,
    index=test_data.index
)

train_design[target_col] = df[target_col].values
train_design[category_col] = df[category_col].values
train_design[client_col] = df[client_col].values

test_design[target_col] = test_data[target_col].values
test_design[category_col] = test_data[category_col].values


## Functions

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def assign_tertiles(y_hat):
    q1 = np.quantile(y_hat, 1/3)
    q2 = np.quantile(y_hat, 2/3)

    groups = np.where(
        y_hat <= q1, 0,
        np.where(y_hat <= q2, 1, 2)
    )

    return groups, q1, q2


def metrics_for_one_complication(y_true, y_pred):
    results = {}

    for cls in [0, 1, 2]:
        tp = np.sum((y_true == cls) & (y_pred == cls))
        fp = np.sum((y_true != cls) & (y_pred == cls))
        fn = np.sum((y_true == cls) & (y_pred != cls))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0 else 0.0
        )

        results[cls] = {
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

    macro_f1 = np.mean([results[cls]["f1"] for cls in [0, 1, 2]])

    return {
        "per_class": results,
        "macro_f1": macro_f1
    }


def centralized_logistic_regression_binary(
    train_df,
    test_df,
    variables,
    target_col,
    category_col,
    epochs=50,
    lr=0.001,
    random_state=42
):
    X_train = train_df[variables].to_numpy(dtype=float)
    y_train = train_df[target_col].to_numpy(dtype=int)

    X_test = test_df[variables].to_numpy(dtype=float)
    y_true_group = test_df[category_col].to_numpy(dtype=int)

    model = SGDClassifier(
        loss="log_loss",
        penalty=None,
        learning_rate="constant",
        eta0=lr,
        max_iter=1,
        tol=None,
        warm_start=True,
        random_state=random_state
    )

    history = []

    # Initialisering
    model.partial_fit(
        X_train[:1],
        y_train[:1],
        classes=np.array([0, 1])
    )

    for epoch in range(epochs):

        old_coef = model.coef_.copy()
        old_intercept = model.intercept_.copy()

        # Én epoch på samlet data
        model.partial_fit(X_train, y_train)

        coef_change = np.linalg.norm(model.coef_ - old_coef)
        intercept_change = np.linalg.norm(model.intercept_ - old_intercept)

        logits = X_test @ model.coef_.T + model.intercept_
        probs = sigmoid(logits).ravel()

        y_pred_group, q1, q2 = assign_tertiles(probs)

        metrics = metrics_for_one_complication(y_true_group, y_pred_group)
        macro_f1 = metrics["macro_f1"]

        history.append({
            "epoch": epoch + 1,
            "macro_f1": macro_f1,
            "q1": q1,
            "q2": q2,
            "coef_change": coef_change,
            "intercept_change": intercept_change
        })

        print(
            f"Epoch {epoch+1:03d} | Macro F1 = {macro_f1:.4f} | "
            f"coef_change = {coef_change:.6f} | "
            f"intercept_change = {intercept_change:.6f}"
        )

    return model.intercept_, model.coef_, pd.DataFrame(history)

## Training

In [9]:
intercept_c, coefs_c, history_c_df = centralized_logistic_regression_binary(
    train_df=train_design,
    test_df=test_design,
    variables=variables,
    target_col=target_col,
    category_col=category_col,
    epochs=50,
    lr=0.001
)

Epoch 001 | Macro F1 = 0.6823 | coef_change = 0.203476 | intercept_change = 3.606441
Epoch 002 | Macro F1 = 0.6977 | coef_change = 0.166967 | intercept_change = 0.566493
Epoch 003 | Macro F1 = 0.7070 | coef_change = 0.145374 | intercept_change = 0.285627
Epoch 004 | Macro F1 = 0.7120 | coef_change = 0.124386 | intercept_change = 0.181893
Epoch 005 | Macro F1 = 0.7143 | coef_change = 0.105656 | intercept_change = 0.131020
Epoch 006 | Macro F1 = 0.7156 | coef_change = 0.089815 | intercept_change = 0.101900
Epoch 007 | Macro F1 = 0.7216 | coef_change = 0.076860 | intercept_change = 0.083268
Epoch 008 | Macro F1 = 0.7209 | coef_change = 0.066442 | intercept_change = 0.070276
Epoch 009 | Macro F1 = 0.7216 | coef_change = 0.058099 | intercept_change = 0.060613
Epoch 010 | Macro F1 = 0.7239 | coef_change = 0.051382 | intercept_change = 0.053087
Epoch 011 | Macro F1 = 0.7213 | coef_change = 0.045917 | intercept_change = 0.047033
Epoch 012 | Macro F1 = 0.7253 | coef_change = 0.041410 | intercep

In [10]:
X_test_final = test_design[variables].to_numpy(dtype=float)
y_true_group = test_design[category_col].to_numpy(dtype=int)

logits = X_test_final @ coefs_c.T + intercept_c
probs = sigmoid(logits).ravel()

y_pred_group, q1, q2 = assign_tertiles(probs)

final_metrics_c = metrics_for_one_complication(y_true_group, y_pred_group)

print("\nCentralized final macro F1:")
print(final_metrics_c["macro_f1"])

print("\nPer class metrics:")
for cls, vals in final_metrics_c["per_class"].items():
    print(f"\nClass {cls}")
    print(vals)

print("\nTertile cutoffs:")
print("q1:", q1)
print("q2:", q2)

history_c_df


Centralized final macro F1:
0.7266001000096796

Per class metrics:

Class 0
{'TP': np.int64(777), 'FP': np.int64(223), 'FN': np.int64(139), 'precision': np.float64(0.777), 'recall': np.float64(0.8482532751091703), 'f1': np.float64(0.8110647181628392)}

Class 1
{'TP': np.int64(634), 'FP': np.int64(366), 'FN': np.int64(464), 'precision': np.float64(0.634), 'recall': np.float64(0.5774134790528234), 'f1': np.float64(0.6043851286939942)}

Class 2
{'TP': np.int64(759), 'FP': np.int64(241), 'FN': np.int64(227), 'precision': np.float64(0.759), 'recall': np.float64(0.7697768762677485), 'f1': np.float64(0.7643504531722054)}

Tertile cutoffs:
q1: 0.0012468821784399114
q2: 0.008766733833422849


,epoch,macro_f1,q1,q2,coef_change,intercept_change
0,1,0.682308,0.024317,0.028926,0.203476,3.606441
1,2,0.697661,0.013059,0.018021,0.166967,0.566493
2,3,0.707004,0.009316,0.014600,0.145374,0.285627
3,4,0.711975,0.007396,0.012991,0.124386,0.181893
4,5,0.714308,0.006255,0.012065,0.105656,0.131020
5,6,0.715627,0.005469,0.011441,0.089815,0.101900
6,7,0.721575,0.004869,0.010958,0.076860,0.083268
7,8,0.720940,0.004380,0.010568,0.066442,0.070276
8,9,0.721581,0.004047,0.010319,0.058099,0.060613
9,10,0.723944,0.003759,0.010064,0.051382,0.053087
